# 🧬 Chromatin ATAC-to-Hi-C Predictor — Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadoliyaP/chromatin-atac-to-hic-predictor/blob/main/demo.ipynb)

This notebook demonstrates how to use the trained model to predict 2D Hi-C contact maps from 1D ATAC-seq signals.

## What this notebook does
1. Downloads pre-processed sample tensors for `chr21`
2. Loads the trained checkpoint
3. Runs evaluation and plots side-by-side comparison (Actual vs Predicted Hi-C)

## 1. Setup

Install dependencies and clone the repository.

In [ ]:
# Install dependencies
!pip install torch numpy matplotlib h5py

# Clone the repository
!git clone https://github.com/VadoliyaP/chromatin-atac-to-hic-predictor.git
%cd chromatin-atac-to-hic-predictor

## 2. Download Sample Data

Download pre-processed sample tensors for `chr21` (validation chromosome).

In [ ]:
import os
import torch
import numpy as np

# Create data directory
os.makedirs('data/chr21', exist_ok=True)

# Note: In a real setup, you would download pre-processed tensors from
# a shared drive or data repository. For this demo, we'll generate
# synthetic data that matches the expected format.

# Generate synthetic ATAC-seq input [1, 100]
np.random.seed(42)
atac_signal = np.random.rand(1, 100).astype(np.float32)

# Generate synthetic Hi-C target [100, 100]
# (distance-dependent decay + some loop structures)
hic_target = np.zeros((100, 100), dtype=np.float32)
for i in range(100):
    for j in range(100):
        # Distance decay
        dist = abs(i - j)
        hic_target[i, j] = np.exp(-dist / 20) * 0.5
        # Add some loop structures
        if 20 <= i <= 30 and 60 <= j <= 70:
            hic_target[i, j] += 0.3
        if 50 <= i <= 60 and 80 <= j <= 90:
            hic_target[i, j] += 0.2

# Apply log1p transform (as in the training pipeline)
hic_target_log = np.log1p(hic_target)

print(f"ATAC-seq input shape: {atac_signal.shape}")
print(f"Hi-C target shape: {hic_target_log.shape}")
print(f"\nATAC-seq range: [{atac_signal.min():.3f}, {atac_signal.max():.3f}]")
print(f"Hi-C target range: [{hic_target_log.min():.3f}, {hic_target_log.max():.3f}]")

## 3. Load Model

Load the trained `DilatedGenomicPredictor` model.

In [ ]:
from hic_predictor_model import DilatedGenomicPredictor

# Initialize model
model = DilatedGenomicPredictor()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Load checkpoint if available
checkpoint_path = 'chromatin_predictor_checkpoint.pt'
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from epoch {checkpoint.get('epoch', '?')}")
else:
    print("No checkpoint found — using random weights for demo")

model.eval()

## 4. Run Prediction

Generate a predicted Hi-C matrix from the ATAC-seq input.

In [ ]:
import torch

# Prepare input tensor [batch=1, channels=1, length=100]
x = torch.from_numpy(atac_signal).unsqueeze(0)

# Run prediction
with torch.no_grad():
    predicted = model(x).squeeze().numpy()

print(f"Predicted shape: {predicted.shape}")
print(f"Predicted range: [{predicted.min():.3f}, {predicted.max():.3f}]")

## 5. Visualization

Plot side-by-side comparison: Actual Hi-C vs Predicted Hi-C.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ATAC-seq signal
axes[0].plot(atac_signal.flatten(), color='steelblue')
axes[0].set_title('ATAC-seq Input (1D)', fontsize=12)
axes[0].set_xlabel('Bin (10 kb)')
axes[0].set_ylabel('Accessibility')

# Actual Hi-C
im1 = axes[1].imshow(hic_target_log, cmap='YlOrRd', interpolation='nearest')
axes[1].set_title('Actual Hi-C (log1p)', fontsize=12)
axes[1].set_xlabel('Bin')
axes[1].set_ylabel('Bin')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Predicted Hi-C
im2 = axes[2].imshow(predicted, cmap='YlOrRd', interpolation='nearest')
axes[2].set_title('Predicted Hi-C', fontsize=12)
axes[2].set_xlabel('Bin')
axes[2].set_ylabel('Bin')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig('demo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nComparison plot saved to demo_comparison.png")

## 6. Evaluation Metrics

Compute correlation metrics between actual and predicted matrices.

In [ ]:
from scipy import stats

# Flatten matrices for correlation
actual_flat = hic_target_log.flatten()
predicted_flat = predicted.flatten()

# Pearson correlation
pearson_r, pearson_p = stats.pearsonr(actual_flat, predicted_flat)

# Spearman correlation
spearman_r, spearman_p = stats.spearmanr(actual_flat, predicted_flat)

# MSE
mse = np.mean((actual_flat - predicted_flat) ** 2)

print("=" * 40)
print("Evaluation Metrics")
print("=" * 40)
print(f"Pearson r:  {pearson_r:.4f} (p={pearson_p:.2e})")
print(f"Spearman ρ: {spearman_r:.4f} (p={spearman_p:.2e})")
print(f"MSE:        {mse:.6f}")
print("=" * 40)

## 7. Next Steps

- Try with real data from ENCODE (see `extract_hic_epigenetics.py`)
- Train on your own cell line data
- Experiment with different window sizes and resolutions

See the [README](https://github.com/VadoliyaP/chromatin-atac-to-hic-predictor) for full documentation.